Einer Barba Abdala 595839

Renata García Morales 612194

Final project: Sports Analytics using Deep Learning Model

Date: 20/05/2026

Materia: Inteligencia Artificial II

*We hereby declare that we have done this activity with academic integrity.*

# **Introduction**

# **Data preparation**

The aerial video to be analyzed for this class' final project has a duration of 2 minutes and 8 seconds. It is recorded at 30 frames per second, which results in a total of 3,840 frames. Those frames have a resolution of 1280 x 720 pixels.

This approach offers advantages over others like scene-change detection or motion-based sampling. Scene-change detection, for example, is not suitable in this situation. This is due to the fact that the video always shows the same aerial view of the field, with no scene changes occurring. Apart from that, elements like the bird crossing the frame would incorrectly provoke scene-change events, introducing unwanted samples in the dataset. Fixed-interval sampling guarantees that frames are obtained from the entire video, capturing different moments of player positions and different ball locations.


There are two class labels belonging to this project, each belonging to a type of tracked object. The annotation format must follow YOLO format, being an integer index. Here, 0 is the player class. This class includes any soccer player that is visible in the frame, and those partially covered with shadows. The second class, of index 1, is the ball class. This class incldues the location of the ball at every frame. Due to its size, tighter, more consistent bounding boxes will be prioritized, to ensure better model accuracy.

In [12]:
import cv2
import ctypes
import os
import shutil
import zipfile
import tempfile
import pandas as pd
from PIL import Image
import yaml
import glob
import numpy as np
from sklearn.model_selection import train_test_split
from IPython.display import Image as IPImage
import ultralytics
from ultralytics import YOLO

In [13]:
VIDEO_PATH = "2023_05_05_15_02_22-players-and-ball-detection.mp4"

cam = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(cam.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cam.get(cv2.CAP_PROP_FPS)
width        = int(cam.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cam.get(cv2.CAP_PROP_FRAME_HEIGHT))

cam.release()

In [14]:
print(total_frames)

3846


Gathering and annotating of data was done in two distinct phases.
First, 192 different frames were selected at a fixed-interval by which one frame is extracted every 20 frames. 
For the second phase, a different 405 frames were selected at a fixed-interval of 4 frames.

This results in a subset of 576 frames distributed throughout the whole video.

In [15]:
SAMPLE_EVERY = 20
SECOND_PASS_EVERY = 4
TARGET_FRAMES = 405

cam = cv2.VideoCapture(VIDEO_PATH)

try:
    if not os.path.exists('dataset/frames'):
        os.makedirs('dataset/frames')
except OSError:
    print('Error: Creating directory of data')

saved = 0

# First phase: sample the whole video every 20 frames.
currentframe = 0
while True:
    ret, frame = cam.read()
    if not ret:
        break
    if currentframe % SAMPLE_EVERY == 0:
        name = f'./dataset/frames/frame{currentframe:05d}.jpg'
        print('Creating... ' + name)
        cv2.imwrite(name, frame)
        saved += 1
    currentframe += 1

# Second phase: sample every 4 frames until 384 total frames are collected.
# Skip multiples of 20 because those were already captured in the first pass.
if saved < TARGET_FRAMES:
    cam.set(cv2.CAP_PROP_POS_FRAMES, 0)
    currentframe = 0

    while saved < TARGET_FRAMES:
        ret, frame = cam.read()
        if not ret:
            break
        if currentframe % SECOND_PASS_EVERY == 0 and currentframe % SAMPLE_EVERY != 0:
            name = f'./dataset/frames/frame{currentframe:05d}.jpg'
            if not os.path.exists(name):
                print('Creating... ' + name)
                cv2.imwrite(name, frame)
                saved += 1
        currentframe += 1

cam.release()
cv2.destroyAllWindows()
print(f'Frames saved: {saved}')

Creating... ./dataset/frames/frame00000.jpg
Creating... ./dataset/frames/frame00020.jpg
Creating... ./dataset/frames/frame00040.jpg
Creating... ./dataset/frames/frame00060.jpg
Creating... ./dataset/frames/frame00080.jpg
Creating... ./dataset/frames/frame00100.jpg
Creating... ./dataset/frames/frame00120.jpg
Creating... ./dataset/frames/frame00140.jpg
Creating... ./dataset/frames/frame00160.jpg
Creating... ./dataset/frames/frame00180.jpg
Creating... ./dataset/frames/frame00200.jpg
Creating... ./dataset/frames/frame00220.jpg
Creating... ./dataset/frames/frame00240.jpg
Creating... ./dataset/frames/frame00260.jpg
Creating... ./dataset/frames/frame00280.jpg
Creating... ./dataset/frames/frame00300.jpg
Creating... ./dataset/frames/frame00320.jpg
Creating... ./dataset/frames/frame00340.jpg
Creating... ./dataset/frames/frame00360.jpg
Creating... ./dataset/frames/frame00380.jpg
Creating... ./dataset/frames/frame00400.jpg
Creating... ./dataset/frames/frame00420.jpg
Creating... ./dataset/frames/fra

These selected frames can then be made into an archive, for upload into the labeling tool.

In [16]:
shutil.make_archive("frames", 'zip', "dataset/frames")

'd:\\Repos\\FinalProjectAi2\\final-project-AI-II\\frames.zip'

# **Data labeling**

To assist in the labeling of the 576 frames belonging to the varied subset, the tool Roboflow was chosen.

This web application offers a simplified, effective process to draw bounding boxes over each individual frame. The annotations generated by this application are consistent with the format needed by the YOLO models.

This tool works by allowing the upload of the selected frames. Then, each frame is presented individually to be annotated through bounding boxes, allowing a different type of box to be used depending on class. Afterwards, the annotated frame can be mark as completed and sent to the Annotated Frames section.

This leads to a full Annotated Frames folder containing the full collection of frames, each with 12 annotated players and 1 annotated ball. There is only one frame where the ball is not visible at all.

This set can then be downloaded, and uploaded to this notebook.

This process begins by loading the exported .zip file directly, and extracting its contents.

In [17]:
with zipfile.ZipFile("SoccerField.yolo26.zip") as zf:
    zf.extractall("dataset")

In [18]:
with open("dataset/data.yaml", "r") as f:
    data = yaml.safe_load(f)

print(data)

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 2, 'names': ['ball', 'player'], 'roboflow': {'workspace': 'renatas-workspace-bae6f', 'project': 'renatas-workspace-bae6f', 'version': 'dataset', 'license': 'Private', 'url': 'https://app.roboflow.com/renatas-workspace-bae6f/renatas-workspace-bae6f/dataset'}}


Afterwards, the filenames of each individual image can be stored.

In [19]:
images = sorted(glob.glob("dataset/**/images/*.jpg", recursive=True))

With this, there is now a full collection of images loaded. The next step is to manually split these images into the necessary categories.

# **Dataset split**

The images in the annotated subset will be split into a 70-15-15 % train-test-validation segmentation. This will allow for the majority of the images to be used for training, while allowing for sufficient testing and validation.

In [20]:
train, temp = train_test_split(images, test_size=0.3, random_state=42)
val, test   = train_test_split(temp,   test_size=0.5, random_state=42)

In [21]:
for split_name, split_files in [("train", train), ("val", val), ("test", test)]:
    os.makedirs(f"split/{split_name}/images", exist_ok=True)
    os.makedirs(f"split/{split_name}/labels", exist_ok=True)
    for img_path in split_files:
        lbl_path = img_path.replace("images", "labels").replace(".jpg", ".txt")
        shutil.copy(img_path, f"split/{split_name}/images/")
        shutil.copy(lbl_path, f"split/{split_name}/labels/")

Once the files are distributed amongs the different categories, the data.yaml file must be updated to reflect this.

In [28]:
data.update({"train": "train/images", "val": "val/images", "test": "test/images"})
with open("split/data.yaml", "w") as f:
    yaml.dump(data, f)

In [29]:
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 387, Val: 83, Test: 83


# **Model training**

In [24]:
ultralytics.checks()

Ultralytics 8.4.47  Python-3.12.1 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2080 SUPER, 8192MiB)
Setup complete  (24 CPUs, 31.9 GB RAM, 1147.7/3726.0 GB disk)


In [25]:
# Run inference on an image with YOLO26n
!yolo predict model=yolo26n.pt source='https://ultralytics.com/images/zidane.jpg'

'yolo' is not recognized as an internal or external command,
operable program or batch file.


In [30]:
# Load pretrained YOLO26n checkpoint
model = YOLO("yolo26n.pt")

# Fine-tune on the custom dataset
results = model.train(
    data="split/data.yaml",
    epochs=100,
    imgsz=1280,
    batch=8,
    workers=4,
    amp=True,
    box=12.0,
    cls=3.0,
    dfl=2.0
)

New https://pypi.org/project/ultralytics/8.4.52 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.47  Python-3.12.1 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2080 SUPER, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=12.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=3.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=split/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, na

KeyboardInterrupt: 

# **Validation and testing**

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict

# Load the trained model
model = YOLO("runs/detect/train-4/weights/best.pt")

# Get test images
test_dir = Path("split/test/images")
test_images = sorted(list(test_dir.glob("*.jpg")))[:10]  # First 10 test images

print(f"Testing on {len(test_images)} unseen images from test set...\n")

# Track detections
ball_detections = defaultdict(list)
player_detections = defaultdict(list)
class_names = {0: "player", 1: "ball"}

for img_path in test_images:
    # Run inference
    results = model.predict(source=str(img_path), conf=0.5, verbose=False)
    result = results[0]
    
    img_name = img_path.name
    
    # Extract detections by class
    for box in result.boxes:
        class_id = int(box.cls.item())
        confidence = box.conf.item()
        
        if class_id == 0:  # Player
            player_detections[img_name].append(confidence)
        elif class_id == 1:  # Ball
            ball_detections[img_name].append(confidence)

# Compute statistics
print("=" * 70)
print("BALL DETECTION ACCURACY (Target: 70%+)")
print("=" * 70)
ball_found = sum(1 for dets in ball_detections.values() if dets)
ball_accuracy = (ball_found / len(test_images)) * 100
print(f"Detected ball in {ball_found}/{len(test_images)} frames: {ball_accuracy:.1f}%")

if ball_detections:
    ball_confidences = [conf for dets in ball_detections.values() for conf in dets]
    print(f"Average ball confidence: {np.mean(ball_confidences):.3f}")
    print(f"Median ball confidence: {np.median(ball_confidences):.3f}")
else:
    print("No balls detected - consider increasing training epochs or adjusting conf threshold.")

print("\n" + "=" * 70)
print("PLAYER DETECTION ACCURACY (Expected: 95%+)")
print("=" * 70)
player_found = sum(1 for dets in player_detections.values() if len(dets) > 0)
player_accuracy = (player_found / len(test_images)) * 100
print(f"Detected players in {player_found}/{len(test_images)} frames: {player_accuracy:.1f}%")

if player_detections:
    player_counts = [len(dets) for dets in player_detections.values() if dets]
    if player_counts:
        print(f"Average players per frame: {np.mean(player_counts):.1f} (expected: ~12)")
        print(f"Median players per frame: {np.median(player_counts):.0f}")
        player_confidences = [conf for dets in player_detections.values() for conf in dets]
        print(f"Average player confidence: {np.mean(player_confidences):.3f}")

print("\n" + "=" * 70)
print("RECOMMENDATIONS")
print("=" * 70)
if ball_accuracy < 70:
    print("⚠ Ball accuracy below 70%. Try:")
    print("  1. Increase epochs (currently 50) to 75-100")
    print("  2. Lower conf threshold (currently 0.5) to 0.3-0.4")
    print("  3. Check annotation quality in split/train/labels/")
    print("  4. Verify ball bounding boxes are not too tight")
else:
    print("✓ Ball accuracy target achieved! Consider deploying this model.")

## **Accuracy Analysis & Testing on Unseen Data**

After training, test the model on your 10 unseen test frames to measure ball detection accuracy. The optimized training above includes:
- **50 epochs** with early stopping (patience=15) for better convergence  
- **640px images** to preserve small ball details  
- **Aggressive augmentation** to handle shaky footage and lighting changes  
- **Validation enabled** to monitor per-epoch metrics  
- **Lower learning rates** (lr0=0.001, lrf=0.01) for stable fine-tuning  
- **Only 5 frozen layers** to allow network adaptation to the ball domain  

Expected results: **70%+ ball detection accuracy** on unseen test frames, with 95%+ player accuracy.

In [ ]:
# Load the best weights saved during training
model = YOLO("runs/detect/train-4/weights/best.pt")

# Evaluate model performance on the validation set
metrics = model.val()

# Report metrics
print(f"mAP50     : {metrics.box.map50:.3f}")
print(f"mAP50-95  : {metrics.box.map:.3f}")
print(f"Precision : {metrics.box.mp:.3f}")
print(f"Recall    : {metrics.box.mr:.3f}")
print(f"Per-class mAP50 : {metrics.box.maps}")

In [ ]:
# Frames not used during training
UNSEEN_FRAMES = [10, 410, 810, 1210, 1610,
                 2010, 2410, 2810, 3210, 3610]

os.makedirs("inference_frames", exist_ok=True)

cam          = cv2.VideoCapture(VIDEO_PATH)
currentframe = 0
saved        = 0

while True:
    ret, frame = cam.read()
    if not ret:
        break
    if currentframe in UNSEEN_FRAMES:
        name = f'inference_frames/frame{currentframe:05d}.jpg'
        cv2.imwrite(name, frame)
        saved += 1
    currentframe += 1

cam.release()
cv2.destroyAllWindows()
print(f"Unseen frames saved: {saved}")

# **Field Live Video Predictions**

In [42]:
import os
import json
import ctypes

import cv2
import numpy as np
from IPython.display import Video, display
from ultralytics import YOLO

OUTPUT_VIDEO_PATH = "predictions/live_analysis.mp4"
GEOMETRY_STORE_DIR = "predictions/saved_geometry"
GEOMETRY_STORE_PATH = os.path.join(GEOMETRY_STORE_DIR, "manual_setup.json")
CONFIDENCE = 0.25
MIN_LANDMARKS = 6
LK_PARAMS = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
)
PLAYER_ALIASES = {"player", "person", "soccer player"}
BALL_ALIASES = {"ball"}

os.makedirs(os.path.dirname(OUTPUT_VIDEO_PATH), exist_ok=True)
os.makedirs(GEOMETRY_STORE_DIR, exist_ok=True)


if "VIDEO_PATH" not in globals():
    raise RuntimeError("VIDEO_PATH is not defined in the notebook.")

if "model" in globals() and model is not None:
    detection_model = model
else:
    detection_model = YOLO("yolo26n.pt")


def build_class_lookup(yolo_model):
    """Return a stable mapping from class id to class name."""
    raw_names = yolo_model.names
    if isinstance(raw_names, dict):
        return {int(class_id): str(class_name) for class_id, class_name in raw_names.items()}
    return {class_id: str(class_name) for class_id, class_name in enumerate(raw_names)}


def find_class_id(class_lookup, aliases):
    """Find the first class id whose name matches one of the aliases."""
    for class_id, class_name in class_lookup.items():
        normalized = class_name.strip().lower()
        if any(alias in normalized for alias in aliases):
            return class_id
    return None


CLASS_LOOKUP = build_class_lookup(detection_model)
PLAYER_CLASS_ID = find_class_id(CLASS_LOOKUP, PLAYER_ALIASES)
BALL_CLASS_ID = find_class_id(CLASS_LOOKUP, BALL_ALIASES)
TRACKED_CLASS_IDS = [class_id for class_id in [PLAYER_CLASS_ID, BALL_CLASS_ID] if class_id is not None]

if PLAYER_CLASS_ID is None:
    raise RuntimeError(f"Could not find a player class in model names: {CLASS_LOOKUP}")
if BALL_CLASS_ID is None:
    raise RuntimeError(f"Could not find a ball class in model names: {CLASS_LOOKUP}")


REGION_COLORS = {
    "field": (255, 255, 255),
    "left_half": (0, 220, 0),
    "right_half": (0, 165, 255),
    "left_penalty": (255, 140, 0),
    "right_penalty": (0, 0, 255),
    "outside_field": (120, 120, 120),
}


def put_text_with_outline(image, text, origin, color, scale=0.55, thickness=2):
    """Draw readable text using a black outline."""
    x, y = int(origin[0]), int(origin[1])
    cv2.putText(image, text, (x + 1, y + 1), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), thickness + 2, cv2.LINE_AA)
    cv2.putText(image, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)


def wrap_text(text, max_chars=68):
    """Wrap text into multiple lines for the instruction banner."""
    words = text.split()
    if not words:
        return [text]

    lines = []
    current_line = words[0]
    for word in words[1:]:
        candidate = f"{current_line} {word}"
        if len(candidate) <= max_chars:
            current_line = candidate
        else:
            lines.append(current_line)
            current_line = word
    lines.append(current_line)
    return lines


def build_click_banner(frame, title, instructions, color):
    """Create the instruction banner and return it with its height."""
    wrapped_instructions = []
    for instruction in instructions:
        wrapped_instructions.extend(wrap_text(instruction))

    banner_height = max(170, 32 + 28 + 22 * len(wrapped_instructions))
    banner = np.full((banner_height, frame.shape[1], 3), 18, dtype=np.uint8)
    put_text_with_outline(banner, title, (20, 30), (255, 255, 255), scale=0.8, thickness=2)

    y = 62
    for line in wrapped_instructions:
        put_text_with_outline(banner, line, (20, y), (220, 220, 220), scale=0.52, thickness=1)
        y += 22

    return banner, banner_height


def render_click_view(frame, points, title, instructions, color):
    """Render the manual-click UI for landmarks and polygons."""
    banner, _ = build_click_banner(frame, title, instructions, color)

    frame_view = frame.copy()
    for index, (x, y) in enumerate(points):
        cv2.circle(frame_view, (x, y), 5, color, -1, cv2.LINE_AA)
        cv2.putText(frame_view, str(index + 1), (x + 8, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)
        if index > 0:
            cv2.line(frame_view, points[index - 1], points[index], color, 2, cv2.LINE_AA)

    if len(points) > 2:
        cv2.line(frame_view, points[-1], points[0], color, 1, cv2.LINE_AA)

    return np.vstack([banner, frame_view])


def get_display_bounds():
    """Return a conservative display size for the current machine."""
    try:
        user32 = ctypes.windll.user32
        width = int(user32.GetSystemMetrics(0)) - 80
        height = int(user32.GetSystemMetrics(1)) - 120
        return max(640, width), max(480, height)
    except Exception:
        return 1280, 720


def fit_view_to_screen(view):
    """Scale a view down so it fits inside the available display."""
    max_width, max_height = get_display_bounds()
    scale = min(max_width / view.shape[1], max_height / view.shape[0], 1.0)
    if scale >= 1.0:
        return view
    resized = cv2.resize(view, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    return resized


def collect_click_sequence(frame, window_name, title, min_points, instructions, color):
    """Collect an ordered sequence of clicked points from the user."""
    points = []
    cancelled = False
    _, banner_height = build_click_banner(frame, title, instructions, color)
    max_width, max_height = get_display_bounds()
    preview_width = frame.shape[1]
    preview_height = banner_height + frame.shape[0]
    display_scale = min(max_width / preview_width, max_height / preview_height, 1.0)
    display_banner_height = int(round(banner_height * display_scale))
    display_frame_height = int(round(frame.shape[0] * display_scale))
    display_frame_width = int(round(frame.shape[1] * display_scale))

    def to_frame_coordinates(x, y):
        frame_x = int(round(x / display_scale))
        frame_y = int(round((y - display_banner_height) / display_scale))
        frame_x = max(0, min(frame.shape[1] - 1, frame_x))
        frame_y = max(0, min(frame.shape[0] - 1, frame_y))
        return frame_x, frame_y

    def mouse_callback(event, x, y, flags, param):
        nonlocal points
        if event == cv2.EVENT_LBUTTONDOWN:
            if y >= display_banner_height and x < display_frame_width and y < display_banner_height + display_frame_height:
                points.append(to_frame_coordinates(x, y))
        elif event == cv2.EVENT_RBUTTONDOWN and points:
            points.pop()

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.setMouseCallback(window_name, mouse_callback)

    try:
        while True:
            view = fit_view_to_screen(render_click_view(frame, points, title, instructions, color))
            cv2.resizeWindow(window_name, view.shape[1], view.shape[0])
            cv2.imshow(window_name, view)
            key = cv2.waitKey(20) & 0xFF

            if key in (13, 10) and len(points) >= min_points:
                break
            if key == 27:
                cancelled = True
                break
            if key in (8, ord("u"), ord("U")) and points:
                points.pop()
            if key in (ord("r"), ord("R")):
                points.clear()
    finally:
        cv2.destroyWindow(window_name)

    if cancelled:
        raise RuntimeError(f"Manual initialization was cancelled while collecting {title.lower()}.")
    if len(points) < min_points:
        raise RuntimeError(f"Expected at least {min_points} points for {title.lower()}, got {len(points)}.")

    return np.asarray(points, dtype=np.float32).reshape(-1, 1, 2)


def collect_landmarks(frame):
    """Collect stable field landmarks on the first frame."""
    return collect_click_sequence(
        frame=frame,
        window_name="Field Landmark Initialization",
        title="Step 1/2: Landmark initialization",
        min_points=MIN_LANDMARKS,
        instructions=[
            "Left click: add stable landmarks (field corners, penalty corners, center line intersections, circle edge points)",
            "Right click or U: undo last point",
            "R: reset all points",
            "Enter: confirm when you have at least 6 points",
            "Esc: cancel setup",
        ],
        color=(0, 255, 255),
    )


def collect_polygon(frame, label, window_name, color):
    """Collect one ordered polygon on the first frame."""
    return collect_click_sequence(
        frame=frame,
        window_name=window_name,
        title=label,
        min_points=4,
        instructions=[
            "Left click: add polygon vertices in order around the region",
            "Right click or U: undo last point",
            "R: reset all points",
            "Enter: confirm when the polygon has at least 4 vertices",
            "Esc: cancel setup",
        ],
        color=color,
    )


def polygon_to_list(polygon):
    """Convert a polygon array to a JSON-friendly list."""
    return polygon.reshape(-1, 2).astype(int).tolist()


def polygon_from_list(points):
    """Convert a JSON-friendly list back into OpenCV polygon format."""
    return np.asarray(points, dtype=np.float32).reshape(-1, 1, 2)


def save_geometry_setup(path, landmarks, regions):
    """Persist the manual setup so it can be reused later."""
    payload = {
        "landmarks": polygon_to_list(landmarks),
        "regions": {name: polygon_to_list(polygon) for name, polygon in regions.items()},
    }
    with open(path, "w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, indent=2)
    print(f"Saved geometry setup to: {path}")


def load_geometry_setup(path):
    """Load a stored manual setup from disk."""
    with open(path, "r", encoding="utf-8") as file_handle:
        payload = json.load(file_handle)

    regions = {name: polygon_from_list(points) for name, points in payload["regions"].items()}
    landmarks = polygon_from_list(payload["landmarks"])
    print(f"Loaded geometry setup from: {path}")
    return landmarks, regions


def choose_geometry_mode():
    """Ask whether to reuse stored polygons or create new ones."""
    if os.path.exists(GEOMETRY_STORE_PATH):
        print("Manual geometry setup found on disk.")
        print(f"Stored file: {GEOMETRY_STORE_PATH}")
        print("1) Create new polygons and landmarks")
        print("2) Use the stored setup")
        choice = input("Choose an option [2]: ").strip() or "2"
        return "stored" if choice == "2" else "new"

    print("No stored geometry setup found yet. A new one will be created.")
    print(f"It will be saved to: {GEOMETRY_STORE_PATH}")
    return "new"


def polygon_to_int32(polygon):
    """Convert a polygon to OpenCV's integer contour format."""
    return np.asarray(polygon, dtype=np.float32).reshape(-1, 1, 2)


def transform_polygon(polygon, homography):
    """Warp a polygon using a homography matrix."""
    if homography is None:
        return polygon.copy()
    warped = cv2.perspectiveTransform(polygon.astype(np.float32), homography)
    return warped


def transform_region_set(initial_regions, homography):
    """Warp all stored field polygons for the current frame."""
    return {name: transform_polygon(polygon, homography) for name, polygon in initial_regions.items()}


def track_landmarks(previous_gray, current_gray, previous_points):
    """Track landmarks with pyramidal Lucas-Kanade optical flow."""
    next_points, status, error = cv2.calcOpticalFlowPyrLK(
        previous_gray,
        current_gray,
        previous_points,
        None,
        **LK_PARAMS,
    )

    if next_points is None or status is None:
        status = np.zeros((previous_points.shape[0], 1), dtype=np.uint8)
        return previous_points.copy(), status

    status_mask = status.reshape(-1).astype(bool)
    tracked_points = previous_points.copy()
    tracked_points[status_mask] = next_points[status_mask]
    return tracked_points.astype(np.float32), status.astype(np.uint8)


def estimate_homography(initial_points, tracked_points, status):
    """Estimate the homography from initial landmarks to current tracked landmarks."""
    valid_mask = status.reshape(-1).astype(bool)
    if int(valid_mask.sum()) < MIN_LANDMARKS:
        return None, valid_mask

    source_points = initial_points[valid_mask].astype(np.float32)
    destination_points = tracked_points[valid_mask].astype(np.float32)
    homography, inlier_mask = cv2.findHomography(source_points, destination_points, cv2.RANSAC)
    return homography, valid_mask


def polygon_centroid(polygon):
    """Return the centroid of a polygon for labels."""
    points = polygon.reshape(-1, 2)
    return tuple(np.mean(points, axis=0).astype(int))


def draw_polygon(image, polygon, color, label):
    """Draw a polygon outline and label."""
    if polygon is None or len(polygon) < 3:
        return image

    contour = polygon.reshape(-1, 1, 2).astype(np.int32)
    cv2.polylines(image, [contour], True, color, 2, cv2.LINE_AA)
    center = polygon_centroid(polygon)
    put_text_with_outline(image, label, (center[0] + 6, center[1] - 6), color, scale=0.55, thickness=2)
    return image


def draw_landmarks(image, tracked_points, status):
    """Draw the tracked landmark points on the frame."""
    status_mask = status.reshape(-1).astype(bool)
    for index, point in enumerate(tracked_points.reshape(-1, 2)):
        x, y = int(point[0]), int(point[1])
        color = (0, 255, 0) if status_mask[index] else (0, 0, 255)
        cv2.circle(image, (x, y), 5, color, -1, cv2.LINE_AA)
        cv2.putText(image, str(index + 1), (x + 6, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return image


def classify_point(point, regions):
    """Classify a point using the transformed field polygons and priority order."""
    px = (float(point[0]), float(point[1]))

    if cv2.pointPolygonTest(regions["field"].reshape(-1, 1, 2).astype(np.float32), px, False) < 0:
        return "outside_field"
    if cv2.pointPolygonTest(regions["left_penalty"].reshape(-1, 1, 2).astype(np.float32), px, False) >= 0:
        return "left_penalty"
    if cv2.pointPolygonTest(regions["right_penalty"].reshape(-1, 1, 2).astype(np.float32), px, False) >= 0:
        return "right_penalty"
    if cv2.pointPolygonTest(regions["left_half"].reshape(-1, 1, 2).astype(np.float32), px, False) >= 0:
        return "left_half"
    if cv2.pointPolygonTest(regions["right_half"].reshape(-1, 1, 2).astype(np.float32), px, False) >= 0:
        return "right_half"

    field_center_x = float(np.mean(regions["field"].reshape(-1, 2)[:, 0]))
    return "left_half" if point[0] < field_center_x else "right_half"


def draw_labeled_box(image, x1, y1, x2, y2, label, color, foot_point=None):
    """Draw a player or ball detection with a label and optional foot point."""
    cv2.rectangle(image, (x1, y1), (x2, y2), color, 2, cv2.LINE_AA)
    if foot_point is not None:
        cv2.circle(image, foot_point, 5, color, -1, cv2.LINE_AA)
    if label:
        put_text_with_outline(image, label, (x1, max(18, y1 - 8)), color, scale=0.55, thickness=2)
    return image


def compose_video_frame(frame, counts, ball_position, panel_height):
    """Attach a summary panel above the annotated frame."""
    output = np.zeros((frame.shape[0] + panel_height, frame.shape[1], 3), dtype=np.uint8)
    output[:panel_height] = (18, 22, 30)
    output[panel_height:] = frame
    cv2.rectangle(output, (0, 0), (frame.shape[1] - 1, panel_height - 1), (255, 255, 255), 1)

    total_players = sum(counts.values())
    top_row_y = 34
    bottom_row_y = 78
    columns = [20, frame.shape[1] // 4 + 20, frame.shape[1] // 2 + 20, 3 * frame.shape[1] // 4 + 20]
    summary_items = [
        (f"Total players detected: {total_players}", columns[0], top_row_y),
        (f"Players in the left half: {counts['left_half']}", columns[1], top_row_y),
        (f"Players in the right half: {counts['right_half']}", columns[2], top_row_y),
        (f"Ball position: {ball_position}", columns[3], top_row_y),
        (f"Players in the left penalty area: {counts['left_penalty']}", columns[0], bottom_row_y),
        (f"Players in the right penalty area: {counts['right_penalty']}", columns[1], bottom_row_y),
        (f"Players outside the court: {counts['outside']}", columns[2], bottom_row_y),
    ]
    for text, x, y in summary_items:
        cv2.putText(output, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (245, 245, 245), 1, cv2.LINE_AA)
    return output


cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

ret, first_frame = cap.read()
if not ret:
    cap.release()
    raise RuntimeError(f"Could not read the first frame from: {VIDEO_PATH}")

first_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)

geometry_mode = choose_geometry_mode()
if geometry_mode == "stored":
    initial_points, initial_regions = load_geometry_setup(GEOMETRY_STORE_PATH)
    print(f"Loaded saved geometry from {GEOMETRY_STORE_PATH}")
else:
    print("Click stable landmarks on the first frame, then press Enter.")
    initial_points = collect_landmarks(first_frame)

    print("Define the field polygon on the first frame, then press Enter.")
    initial_field_polygon = collect_polygon(
        first_frame,
        label="Step 2/2: Field polygon",
        window_name="Field Polygon",
        color=(255, 255, 255),
    )

    print("Define the left half polygon on the first frame, then press Enter.")
    initial_left_half_polygon = collect_polygon(
        first_frame,
        label="Left half polygon",
        window_name="Left Half Polygon",
        color=(0, 220, 0),
    )

    print("Define the right half polygon on the first frame, then press Enter.")
    initial_right_half_polygon = collect_polygon(
        first_frame,
        label="Right half polygon",
        window_name="Right Half Polygon",
        color=(0, 165, 255),
    )

    print("Define the left penalty polygon on the first frame, then press Enter.")
    initial_left_penalty_polygon = collect_polygon(
        first_frame,
        label="Left penalty polygon",
        window_name="Left Penalty Polygon",
        color=(255, 140, 0),
    )

    print("Define the right penalty polygon on the first frame, then press Enter.")
    initial_right_penalty_polygon = collect_polygon(
        first_frame,
        label="Right penalty polygon",
        window_name="Right Penalty Polygon",
        color=(0, 0, 255),
    )

    initial_regions = {
        "field": initial_field_polygon,
        "left_half": initial_left_half_polygon,
        "right_half": initial_right_half_polygon,
        "left_penalty": initial_left_penalty_polygon,
        "right_penalty": initial_right_penalty_polygon,
    }
    save_geometry_setup(GEOMETRY_STORE_PATH, initial_points, initial_regions)
    print(f"Saved geometry setup to {GEOMETRY_STORE_PATH}")

previous_points = initial_points.copy()

current_homography = np.eye(3, dtype=np.float64)
current_regions = transform_region_set(initial_regions, current_homography)

writer = cv2.VideoWriter(
    OUTPUT_VIDEO_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (frame_width, frame_height + 110),
)


def process_frame(frame, regions, tracked_points, tracked_status):
    """Predict detections and annotate the frame with regions and labels."""
    predict_kwargs = {"conf": CONFIDENCE, "verbose": False}
    if TRACKED_CLASS_IDS:
        predict_kwargs["classes"] = TRACKED_CLASS_IDS

    result = detection_model.predict(frame, **predict_kwargs)[0]
    annotated = frame.copy()
    counts = {
        "left_half": 0,
        "right_half": 0,
        "left_penalty": 0,
        "right_penalty": 0,
        "outside": 0,
    }
    ball_position = "not detected"

    for region_name in ["field", "left_half", "right_half", "left_penalty", "right_penalty"]:
        annotated = draw_polygon(annotated, regions[region_name], REGION_COLORS[region_name], region_name)

    annotated = draw_landmarks(annotated, tracked_points, tracked_status)

    boxes = result.boxes
    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            class_id = int(box.cls.item())
            class_name = str(result.names[class_id]).lower() if isinstance(result.names, (list, tuple)) else str(result.names.get(class_id, class_id)).lower()
            confidence = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            x1_i, y1_i, x2_i, y2_i = int(x1), int(y1), int(x2), int(y2)

            if class_id == PLAYER_CLASS_ID or "player" in class_name or "person" in class_name:
                center_x = int((x1_i + x2_i) / 2)
                center_y = int((y1_i + y2_i) / 2)
                region_label = classify_point((center_x, center_y), regions)
                color = REGION_COLORS.get(region_label, (255, 255, 255))
                if region_label in counts:
                    counts[region_label] += 1
                draw_labeled_box(
                    annotated,
                    x1_i,
                    y1_i,
                    x2_i,
                    y2_i,
                    "",
                    color,
                    foot_point=(center_x, center_y),
                )
            elif class_id == BALL_CLASS_ID or "ball" in class_name:
                ball_center = (int((x1_i + x2_i) / 2), int((y1_i + y2_i) / 2))
                ball_position = "left half" if ball_center[0] < regions["field"].reshape(-1, 2)[:, 0].mean() else "right half"
                draw_labeled_box(
                    annotated,
                    x1_i,
                    y1_i,
                    x2_i,
                    y2_i,
                    "ball",
                    (0, 255, 255),
                    foot_point=ball_center,
                )

    return annotated, result, counts, ball_position


# Process the first frame immediately with identity homography.
first_tracked_points = previous_points.copy()
first_status = np.ones((initial_points.shape[0], 1), dtype=np.uint8)
annotated_first_frame, first_result, first_counts, first_ball_position = process_frame(first_frame, current_regions, first_tracked_points, first_status)
writer.write(compose_video_frame(annotated_first_frame, first_counts, first_ball_position, 110))

previous_gray = first_gray.copy()
frame_index = 1
print("Processed frame 1")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        current_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        tracked_points, tracked_status = track_landmarks(previous_gray, current_gray, previous_points)
        homography, valid_mask = estimate_homography(initial_points, tracked_points, tracked_status)

        if homography is not None:
            current_homography = homography

        current_regions = transform_region_set(initial_regions, current_homography)
        annotated_frame, result, counts, ball_position = process_frame(frame, current_regions, tracked_points, tracked_status)
        writer.write(compose_video_frame(annotated_frame, counts, ball_position, 110))

        previous_points = tracked_points
        previous_gray = current_gray
        frame_index += 1

        if frame_index % 100 == 0:
            print(f"Processed {frame_index} frames")
finally:
    cap.release()
    writer.release()
    cv2.destroyAllWindows()

print(f"Saved annotated video to {OUTPUT_VIDEO_PATH}")
display(Video(OUTPUT_VIDEO_PATH, embed=False))

Manual geometry setup found on disk.
Stored file: predictions/saved_geometry\manual_setup.json
1) Create new polygons and landmarks
2) Use the stored setup
Loaded geometry setup from: predictions/saved_geometry\manual_setup.json
Loaded saved geometry from predictions/saved_geometry\manual_setup.json
Processed frame 1
Processed 100 frames
Processed 200 frames
Processed 300 frames
Processed 400 frames
Processed 500 frames
Processed 600 frames
Processed 700 frames
Processed 800 frames
Processed 900 frames
Processed 1000 frames
Processed 1100 frames
Processed 1200 frames
Processed 1300 frames
Processed 1400 frames
Processed 1500 frames
Processed 1600 frames
Processed 1700 frames
Processed 1800 frames
Processed 1900 frames
Processed 2000 frames
Processed 2100 frames
Processed 2200 frames
Processed 2300 frames
Processed 2400 frames
Processed 2500 frames
Processed 2600 frames
Processed 2700 frames
Processed 2800 frames
Processed 2900 frames
Processed 3000 frames
Processed 3100 frames
Processe

# **Predictions**

# **Referencias**
EXTRACCION DE DATOS:
https://www.geeksforgeeks.org/python/extract-images-from-video-in-python/
https://arxiv.org/html/2408.03340v1

DATA LABELING>
https://docs.ultralytics.com/es/integrations/roboflow/
https://docs.ultralytics.com/es/integrations/roboflow/#roboflow-collect
https://blog.roboflow.com/tips-for-how-to-label-images/


PARA EL MODELO:
https://docs.ultralytics.com/tasks/detect/
https://colab.research.google.com/github/ultralytics/ultralytics/blob/main/examples/tutorial.ipynb#scrollTo=zR9ZbuQCH7FX
https://docs.ultralytics.com/modes/train/
https://docs.ultralytics.com/datasets/detect/coco/